# Дополнительное задание

Используется `ChatPromptTemplate` и структурированный JSON-вывод для извлечения дополнительных полей.

## Шаг 1. JSON-схема и ChatPromptTemplate

В системном сообщении заданы правила извлечения, в пользовательском сообщении подставляется текст заявки.

In [ ]:
import json
import os
import re
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_gigachat.chat_models import GigaChat

load_dotenv()
GIGA_KEY = os.getenv("GIGA_KEY")

json_schema = {
    "type": "object",
    "properties": {
        "count_adults": {"type": "integer"},
        "count_children": {"type": "integer"},
        "start_date": {"type": "string", "description": "Дата в формате YYYY-MM-DD"},
        "nights": {"type": "integer"},
        "price_per_day": {"type": "integer"},
        "remarks": {"type": "string"},
    },
    "required": ["count_adults", "count_children", "start_date", "nights", "price_per_day", "remarks"],
}

parser = JsonOutputParser(schema=json_schema)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Ты эксперт по анализу заявок на аренду жилья. "
        "Извлекай структурированные поля строго по смыслу текста. "
        "При диапазоне дат бери самую раннюю дату заезда, при диапазоне ночей или дней бери максимум, "
        "при диапазоне цены бери максимум. Семья без уточнений обычно означает 2 взрослых.",
    ),
    (
        "human",
        "Заявка: {text}\n\nВерни JSON по инструкции:\n{format_instructions}",
    ),
])

llm = GigaChat(
    credentials=GIGA_KEY,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.1,
    max_tokens=1200,
) if GIGA_KEY else None

structured_chain = prompt | llm | parser if llm else None


## Шаг 2. Локальный fallback

Он нужен только для проверки ноутбука без ключа GigaChat.

In [ ]:
MONTHS = {
    "январ": "01", "феврал": "02", "март": "03", "апрел": "04", "ма": "05", "июн": "06",
    "июл": "07", "август": "08", "сентябр": "09", "октябр": "10", "ноябр": "11", "декабр": "12",
}

def fallback_structured_extract(text: str) -> dict:
    lower = text.lower()
    word_numbers = {
        "один": 1, "одного": 1, "двое": 2, "два": 2, "две": 2, "двух": 2,
        "трое": 3, "три": 3, "четверо": 4, "четыре": 4, "пять": 5, "пяти": 5,
    }
    children = 0
    adults = 2 if "семья" in lower or "пара" in lower else 1
    child_match = re.search(r"с\s+(\w+)\s+(реб|дет)", lower)
    if child_match:
        children = word_numbers.get(child_match.group(1), 0)
    if "мама с двумя детьми" in lower:
        adults, children = 1, 2
    if "отец и ребенок" in lower:
        adults, children = 1, 1
    if "младенц" in lower:
        adults, children = 2, 1
    if "двое взрослых и двое детей" in lower:
        adults, children = 2, 2
    if "четверо коллег" in lower:
        adults, children = 4, 0
    if "трое студентов" in lower:
        adults, children = 3, 0
    if "пять туристов" in lower:
        adults, children = 5, 0
    if "два взрослых" in lower or "двое взрослых" in lower:
        adults = 2
    if "один взрослый" in lower or "один человек" in lower:
        adults = 1

    date = "unknown"
    numeric_date = re.search(r"(\d{1,2})[.](\d{2})", lower)
    if numeric_date:
        day, month = numeric_date.groups()
        year = "2026" if int(month) >= 6 else "2027"
        date = f"{year}-{month}-{int(day):02d}"
    else:
        for key, month in MONTHS.items():
            m = re.search(r"(\d{1,2})\s+" + key, lower)
            if m:
                year = "2026" if int(month) >= 6 else "2027"
                date = f"{year}-{month}-{int(m.group(1)):02d}"
                break

    night_values = [int(x) for x in re.findall(r"(\d+)\s*(?:ноч|дн)", lower)]
    nights = max(night_values) if night_values else 7 if "недел" in lower else None

    prices = [int(x) for x in re.findall(r"(\d{4,5})", lower)]
    price = max(prices) if prices else None

    remarks = []
    for marker in ["кроват", "кош", "собак", "кот", "парков", "кухн", "метро", "парк", "тихий", "поздний выезд", "раздельные кровати", "стиральная машина", "центр"]:
        if marker in lower:
            remarks.append(marker)

    return {
        "count_adults": adults,
        "count_children": children,
        "start_date": date,
        "nights": nights,
        "price_per_day": price,
        "remarks": ", ".join(remarks) if remarks else "без особых пожеланий",
    }


## Шаг 3. Обработка 15 размеченных заявок

In [ ]:
df_extra = pd.read_csv("rental_extra_markup.csv", sep=";")

predictions = []
for text in df_extra["text"]:
    if structured_chain:
        predictions.append(structured_chain.invoke({"text": text, "format_instructions": parser.get_format_instructions()}))
    else:
        predictions.append(fallback_structured_extract(text))

pred_df = pd.json_normalize(predictions).add_prefix("pred_")
result_df = pd.concat([df_extra, pred_df], axis=1)
result_df.to_csv("rental_extra_with_results.csv", index=False, encoding="utf-8-sig")
result_df.head()


## Шаг 4. Точность по полям 1-4 и средняя точность

In [ ]:
metric_fields = ["count_adults", "count_children", "start_date", "nights", "price_per_day"]

scores = {}
for field in metric_fields:
    expected = result_df[field].astype(str)
    predicted = result_df[f"pred_{field}"].astype(str)
    scores[field] = (expected == predicted).mean()

average_accuracy = sum(scores.values()) / len(scores)

for field, score in scores.items():
    print(f"{field}: {score:.1%}")
print(f"Средняя точность: {average_accuracy:.1%}")

result_df[["text", *metric_fields, *(f"pred_{field}" for field in metric_fields)]]
